In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from IPython.display import Latex, HTML, Math, display
from uncertainties import ufloat
from uncertainties.umath import sqrt
from uncertainties import unumpy as unp
from scipy.stats import linregress
from scipy.optimize import curve_fit
from uncertainties.umath import sin, radians 
from uncertainties.umath import *

In [2]:
# 1. Wärmekraftmaschine

#Konstanten (zur Einstellung von Cassy) - wie genau wir das einstellen wird improvisiert?
A = 28.3                #Querschnittsfläche [cm²]


#Messwerte für ideal und unbelastet
f_ll_werte = np.array([69.98, 69.04, 69.09, 70.10, 68.77, 65.59, 66.95, 64.94, 68.85, 66.39])/10            #Leerlauffrequenz [Hz]; zehnmal für die zehnfach Frequenz messen
f_ll_mean = np.mean(f_ll_werte)                  #mittelwert bestimmen --> Leerlauffrequenz [Hz] (auch zum kalibrieren von Cassy?)
f_ll_u = np.std(f_ll_werte, ddof=1)/np.sqrt(len(f_ll_werte))            #standardabweichung vom mittelwert = unsicherheit
f_ll = ufloat(f_ll_mean, f_ll_u)
print(rf"Leerlauffrequenz: {f_ll}")

W_werte = np.array([19660, 19920, 19470, 19630, 20690, 20460, 20520, 20860, 20880, 20230])/(10**4)          #Arbeit: Fläche in Cassy herausfinden [hPacm³ --> Pam³]; Messreihe von 10 Messungen
W_mean = np.mean(W_werte)                        #mittelwert bestimmen für richtigen Wert; umrechnen in [Pam³]!!
W_u = np.std(W_werte, ddof=1)/np.sqrt(len(W_werte))             #standardabweichung vom mittelwert = unsicherheit
f_ll = ufloat(f_ll_mean, f_ll_u)
W = ufloat(W_mean, W_u)
print(rf"Arbeit: {W}")

T_1 = ufloat(123+273.15, 10)                     #Temperatur T1 (warm) abgeschätzt und umgerechnet [K]
T_2 = ufloat(18+273.15, 10)                     #Temperatur T2 (kalt) abgeschätzt und umgerechnet [K]

U = ufloat(12.33, 12.33*0.003 + 0.03)                              #Spannung an der Quelle ablesen [V]
A = ufloat(13.93, 13.93*0.005 + 0.03)                              #Stromstärke an der Quelle ablesen [A] 
print(rf"Spannung: {U}, Stromstärke: {A}")

P_zu = U * A                                #zugeführte Lesistung [W]
print(rf"zugeführte Leistung: {P_zu}")


eta_ideal = (T_1 - T_2) / T_1               #idealer Wirkungsgrad (einheitslos)
print(rf"idealer Wirkungsgrad: {eta_ideal}")

eta_unbelastet = (f_ll * W) / P_zu          #unbelasteter Wirkungsgrad (einheitslos)
print(rf"unbelasteter Wirkungsgrad: {eta_unbelastet}")
print()


#Messwerte für belastet mit Bremszaum
r_u = ufloat(0.256, 0.001)                              #radius Bremszaum [m]
r = 0.256

m = unp.uarray([63, 66, 64, 66, 65], 2)/1000                              #Gewicht Federwaage [kg]
F_u = 9.81 * m                                #Kraft [N]
F = unp.nominal_values(F_u)
print(rf"Kraft: {F}")

f_brems_werte = np.array([43.59, 47.34, 41.16, 50.63, 48.80])/10         #frequenz mit Bremszaum; wieder 10 mal messen [Hz]
omega_werte = 2 * np.pi * f_brems_werte                 #Kreisfrequenz [1/s]
print(rf"Kreisfrequenz: {omega_werte}")                 #nur nominal werte. der rest wird später mit mittelwert und standardabweichung gerechnet

P_motor_werte = r * F * omega_werte
print(rf"Motorleistungswerte: {P_motor_werte}")         #einzelne werte für die motorleisung; messreihe n=5

P_motor_mean = np.mean(P_motor_werte)                     #Leistung Motor [W]; Mittelwert
P_motor_u = np.std(P_motor_werte, ddof=1)/np.sqrt(len(P_motor_werte))           #Standardabweichung vom Mittelwert = Unsicherheit
P_motor = ufloat(P_motor_mean, P_motor_u)
print(rf"Motorleistung mittelwert: {P_motor}")


eta_belastet = P_motor / P_zu               #bealsteter Wirkungsgrad (einheitslos)
print(rf"belasteter Wirkungsgrad: {eta_belastet}")
print()


#Berechnung mechanischer Wirkungsgrad vom Motor
eta_motor = P_motor / (f_ll * W)            #mechanischer Wirkungsgrad vom Motor (einheitslos)
print(rf"mechanischer Wirkungsgrad Motor: {eta_motor}")



Leerlauffrequenz: 6.80+/-0.06
Arbeit: 2.023+/-0.017
Spannung: 12.33+/-0.07, Stromstärke: 13.93+/-0.10
zugeführte Leistung: 171.8+/-1.5
idealer Wirkungsgrad: 0.265+/-0.031
unbelasteter Wirkungsgrad: 0.0801+/-0.0012

Kraft: [0.61803 0.64746 0.62784 0.64746 0.63765]
Kreisfrequenz: [27.38840475 29.74459924 25.86159072 31.81176721 30.6619443 ]
Motorleistungswerte: [4.33327508 4.93016019 4.15665693 5.27279278 5.00520673]
Motorleistung mittelwert: 4.74+/-0.21
belasteter Wirkungsgrad: 0.0276+/-0.0013

mechanischer Wirkungsgrad Motor: 0.345+/-0.016


In [3]:
# 2. Kältemaschine

#Werte Elektromotor (Annahme: wirkungsgrad 100%)
U_em = ufloat(233.2, 233.2*0.006 + 0.2)                           #Spannung zum Betreiben des EM [V]
A_em = ufloat(0.27, 0.27*0.015)                           #Stromstärke am EM [A]
print(rf"Spannung EM: {U_em}, Stromstärke EM: {A_em}")

P_em = U_em * A_em                          #Leistung EM [W]
print(rf"Leitsung Elektromotor: {P_em}")
print()

#Werte Gegenheizung
U_geg = ufloat(5.14, 5.14*0.003 + 0.03)                          #Spannung an der Gegenheizung [V]
A_geg = ufloat(1.23, 1.23*0.005 + 0.03)                          #Stromstärke an der Gegenheizung [A]
print(rf"Spannung Gegenheizung: {U_geg}, Stromstärke Gegenheizung: {A_geg}")

P_geg = U_geg * A_geg                       #Leistung an der Gegenheizung [W]
print(rf"Leistung Gegenheiziung: {P_geg}")
print()


#Berechnung Wirkungsgrad
eta_kalt = P_geg / P_em                     #Wirkungsgrad der Kältemaschine
print(rf"Wirkungsgrad Kältemaschine: {eta_kalt}")

Spannung EM: 233.2+/-1.6, Stromstärke EM: 0.270+/-0.004
Leitsung Elektromotor: 63.0+/-1.0

Spannung Gegenheizung: 5.14+/-0.05, Stromstärke Gegenheizung: 1.23+/-0.04
Leistung Gegenheiziung: 6.32+/-0.19

Wirkungsgrad Kältemaschine: 0.1004+/-0.0035
